# GoPreaux Example

In this notebook we look at how we can import and use models from [GoPreaux (caat)](https://github.com/crpellegrino/gopreaux), a package for using multi-dimensional Gaussian Process Regression to simulate extragalactic astronomical transient light curves.  This model requires that the GoPreaux (caat) package is installed. Currently GoPreaux is not available on PyPI, so users will need to install it from source from https://github.com/crpellegrino/gopreaux.

**Dependency Conflicts:**

There is a version conflict with numpy between GoPreaux and LightCurveLynx, but this does not impact
the functions we need. Users can install GoPreax first and then install LightCurveLynx (upgrading all
dependencies). You will still get errors about the version requirements for caat, but they can be ignored.

## Load a GoPreaux Model

LightCurveLynx's GoPreaux wrapper uses pretrained models that GoPreaux stores in fits files. Users can find these models at: https://github.com/crpellegrino/gopreaux/tree/main/data/final_models. This notebook requires you to download the files, save them to a local directory, and update the `filename` variable to point to the file.

In [ ]:
from caat import SNModel
from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR

filename = _LIGHTCURVELYNX_BASE_DATA_DIR / "model_files" / "SESNe_SNIIb_GP_model.fits"
gp_model = SNModel(str(filename))

GoPreaux models the difference in brightness (delta magnitude) from the peak of the light curve and the wavelength closest to the V-band. In other words, for each query time and wavelength we want to predict, GoPreaux will provide a delta from a reference magnitude.  To fully define the light curve, we provide a combination of the GoPreaux and a reference magnitude.

In [ ]:
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.models.gopreaux_models import GoPreauxModel
from lightcurvelynx.utils.extrapolate import LastValue

# Sample the brightness randomly from 15 to 18 mag.
brightness_sampler = NumpyRandomFunc("uniform", low=15, high=18)

mjd_start = 60676.0

model = GoPreauxModel(
    gp_model,
    intrinsic_brightness=brightness_sampler,
    # Standard attributes for all physical objects, set all to constants
    # for this example.
    ra=45.0,
    dec=-20.0,
    redshift=0.1,
    t0=mjd_start + 100.0,
    node_label="source",
    time_extrapolation=LastValue(),
    wave_extrapolation=LastValue(),
)

The model will automatically use the bounds defined by the underlying gopreaux model.

In [ ]:
print(f"Wavelengths (angstroms): [{model.minwave()}, {model.maxwave()}]")
print(f"Phases (days): [{model.minphase()}, {model.maxphase()}]")

## Create the Survey Data

There are two types of data files we need to run a LightCurveLynx simulation an `ObsTable` and passband information. In most cases we will want to load the simulated or actual observation tables from various surveys.

In this example we will use the default Rubin passbands. But we will create a fake observation cadence that is better for visualizing this specific object. Specifically, we will take make a fake `OpSim` that observes the exact same location on the sky (45.0, -20.0) three times a night, once in each of the 'r", "g", and 'i' bands. This is particularly realistic for a survey, but provides densely sampled output curves for the purpose of this notebook. For examples of more realistic RA/dec distributions and opsims, see the other notebooks.

Note that we specify a lot of noise parameters with the fake `OpSim`, but in normal cases users will be loading a predefined opsim with all of that information pre-populated.

In [ ]:
import numpy as np

from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.obstable.opsim import OpSim

filters = ["g", "r", "i"]
passband_group = PassbandGroup.from_preset(preset="LSST", filters=filters)

num_days = 200
mjd_start = 60676.0
mjd_end = mjd_start + num_days

num_samples = 50
survey_data = {
    "observationStartMJD": np.linspace(mjd_start, mjd_end, num_samples),
    "fieldRA": np.full(num_samples, 45.0),
    "fieldDec": np.full(num_samples, -20.0),
    "zp_nJy": np.random.normal(loc=1.0, scale=0.1, size=num_samples),
    "filter": [filters[i % 3] for i in range(num_samples)],
    "seeing": np.random.normal(loc=1.1, scale=0.1, size=num_samples),
    "skybrightness": np.random.normal(loc=20.0, scale=0.1, size=num_samples),
    "exptime": np.full(num_samples, 30.0),
    "nexposure": np.full(num_samples, 2),
}
obs_table = OpSim(survey_data)

## Generate the simulations

We can now generate random simulations with all the information defined above. The `simulate_lightcurves` function takes four parameters: the source from which we want to sample (here the collection of lightcurves), the number of results to simulate (100), the opsim, and the passband information. We also run simulations in parallel, using four processes and 25 light curves per simulation batch.

In [ ]:
from lightcurvelynx.simulate import simulate_lightcurves

lightcurves = simulate_lightcurves(
    model,
    5,
    obs_table,
    passband_group,
)

The results are written in the [nested-pandas](https://github.com/lincc-frameworks/nested-pandas) format for easy analysis. Each row corresponds to a single simulated object, with a unique id, ra, dec, etc. The column `params` include all internal state, including hyperparameter settings, that was used to generate this object. The nested `lightcurve` column contains the times, filters, and fluxes for each observation of that object.  We can treat it as a (nested) table.

Let's look at the lightcurve for the first object sampled:

In [ ]:
print(lightcurves.loc[0]["lightcurve"])

Let's plot the first few lightcurves to see what they look like when observed via Rubin's cadence. We plot the results in magnitudes to be consistent with EZTaoX outputs. However note that our error bars can be huge for small fluxes (and this is something we need to fix). 

In [ ]:
for idx in range(3):
    # Extract the row for this object.
    lc = lightcurves.loc[idx]

    if lc["nobs"] == 0:
        continue

    # Unpack the nested columns (filters, mjd, flux, and flux error).
    lc_filters = np.asarray(lc["lightcurve"]["filter"], dtype=str)
    lc_mjd = np.asarray(lc["lightcurve"]["mjd"], dtype=float)
    lc_flux = np.asarray(lc["lightcurve"]["flux"], dtype=float)
    lc_fluxerr = np.asarray(lc["lightcurve"]["fluxerr"], dtype=float)

    scale = np.exp(lc["params"]["source.eztaox_log_kernel_param_0"])
    sigma = np.exp(lc["params"]["source.eztaox_log_kernel_param_1"])
    redshift = lc["params"]["source.redshift"]

    # Plot the lightcurves.
    ax = plot_lightcurves(
        fluxes=lc_flux,
        times=lc_mjd,
        fluxerrs=lc_fluxerr,
        filters=lc_filters,
        title=f"Sample {idx}: Scale={scale:.2f}, Sigma={sigma:.3f}, z={redshift:.3f}",
        plot_magnitudes=True,
    )
    ax.legend()
    plt.show()